# Query Translation
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some way as to improve retrival.

Semantic search on embeddings is hard to get right. Embedding long documents is especially challenging. User queries are a challenge too. If the user provides an ambigious query, they'll end up get an ambiguous matches from embeddings and consequently an ambguous answer. The ambiguous matches land up in the LLM's context from which comes the LLM's response, which could lead to hallucinations.

In this notebook, we'll discuss the following techniques, including what each technique does and when to use them:
1. Multi Query
2. RAG Fusion
3. Query Decomposition
4. Step-back Prompting
5. HYDE (**HY**pothetical **D**ocument **E**mbedding)

<center>
<img src="../images/rag_query_translation.png" width="800" height="480"/>
</center>

In this notebook, we will develop all these techniques on a vectorized content of a web-page. We'll be using the LangChain framework with Google Gemini 2.5 Flash LLM and Cohere embeddings in all examples here. You can use any LLM and any embedding of your choice though - LangChain makes that really easy. Note: use the same LLM and same embedding across all techniques. 

## Multi-Query

**What is does**

* Takes the **user query** and asks the LLM to **generate several alternative** versions of that query.
* The goal is to **capture synonyms, different phrasings, and other angles** of the question.
* _Each expandeod query_ is sent to the vector store → all results are merged → RAG runs on combined context.
* The intuition is that by asking the LLM the same question in N different ways, we will get more relevents chunks of data into the context, thereby improving overall response.

**Why use it?**

Different phrasings capture different embeddings → retrieve more relevant chunks. Helps reduce “embedding mismatch” (e.g., synonyms, domain-specific terms).

**Example:**

User asks: _"How do I cook pasta quickly?"_
LLM generates (3 variations in this case):
* "fast ways to prepare pasta"
* "quick pasta cooking methods"
* "rapid spaghetti preparation"

All run → retrieve docs covering microwaving, pressure cooker, etc. (there could be duplicates, so generate a unique list of retrievals). The retriever might otherwise miss some if only the original query was used.

**Key point:** Multi-query **improves recall** by broadening query formulations.

**Subtle difference**
* Multi-Query **does not** break the question into sub-questions.
* It simply **generates alternative rewrites** of the same question.

The diagram below illustrates this technique.

<center>
<img src="../images/multi_query.png" width="900" height="300"/>
</center>

In [1]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [3]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
# No longer using Google Embeddings as I have apparently exhausted my free quota :(
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/text-embedding-004", task_type="retrieval_document"
# )
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_rag_weng"

In [5]:
def create_or_load_embeddings(
    embeddings, faiss_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        # text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        #     chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        # )
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [6]:
retriever = create_or_load_embeddings(embeddings, faiss_store)

Loading existing embeddings from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\advanced RAG\..\faiss_index_rag_weng


In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# from langchain_google_genai import ChatGoogleGenerativeAI

# Multi Query: Different formulations of same query
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. 

Original question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(template)

In [8]:
generate_queries = (
    prompt_perspectives | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke({"question": "What is task decomposition for LLM agents?"})

['Define task decomposition for large language model agents.',
 'Why do LLM agents employ task decomposition?',
 'How is task decomposition implemented or performed by LLM agents?',
 'What are the advantages of using task decomposition in LLM agent systems?',
 'Describe the process of breaking down complex problems for AI agents utilizing large language models.']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

In [9]:
from langchain.load import dumps, loads


def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs"""
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

In [10]:
# Retrieve
question = "What is task decomposition for LLM agents?"
# here we are firing multiple queries against the vector store, getting all the
# responses & creating a unique set from all the responses.
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question": question})
print(f"Got {len(docs)} documents")
for i, doc in enumerate(docs):
    console.print(
        Markdown(f"### Document {i+1}\n{doc.page_content[:50] + "..."}\n---\n")
    )

# and print the retrival chain too
console.print(f"Retrieval chain: {retrieval_chain}")

Got 10 documents
                                                       Document 1                                                       


                                 Building agents with LLM (large language model) as...                                  
                                                       Document 2                                                       


                                 Given the user request and the call command, the A...                                  
                                                       Document 3                                                       


                                 Challenges in long-term planning and task decompos...                                  
                                                       Document 4                                                       


                                 Boiko et al. (2023) also looked into LLM-empowered...                                  
       

C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_2872\3928205432.py:11: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


In [11]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

multi_query_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = multi_query_rag_chain.invoke({"question": question})
console.print(f"[blue]Question: {question}[/blue]\n")
console.print(f"[yellow]AI Response:[/yellow]")
console.print(Markdown(response))

Question: What is task decomposition for LLM agents?

AI Response:
Task decomposition for LLM agents is the process where the agent breaks down large, complex tasks into smaller, more manageable subgoals. This enables the efficient handling of these complex tasks.

LLMs can perform task decomposition in several ways:

 1 With simple prompting: For example, by using prompts like "Steps for XYZ.\n1." or "What are the subgoals for achievin
 2 By using task-specific instructions: Such as "Write a story outline." for writing a novel.                           
 3 With human inputs.                                                                                                   

In an LLM-powered autonomous agent system, the LLM functions as the agent's brain, parsing user requests into multiple tasks during the task planning stage. Each task is associated with attributes like task type, ID, dependencies, and arguments.


## RAG Fusion
(Think of it as **Multi-Query + ranking / scoring**)

**What is it?**

A **retrieval re-ranking technique** inspired by “Reciprocal Rank Fusion” (RRF) in information retrieval. 
* Generates multiple alternative queries → retrieves documents (multiple retrievals) → uses an algorithm (e.g., Reciprocal Rank Fusion) to rank documents more intelligently.
* You then **fuse/combine the rankes lists/results** from all queries using statistical fusion into one final ranking, _not naive concatenation_ (as you did in Multi-query).

**Why it’s better than Multi-Query**
* **Multi-Query**: retrieve from multiple queries → concatenate.
* **RAG Fusion**: =retrieve → rank → select best documents.

**Key idea**
* **Multiple phrasings + smart ranking → high recall AND high precision**.

**Subtle difference with Multi-Query**
* Both do query expansion, 
* But **RAG Fusion adds mathematical ranking, avoiding irrelevant noise**.

**How it works:**
* Each retrieval returns a ranked list (doc A rank=1, doc B rank=2, etc).
* Fusion scores docs by combining their **reciprocal ranks**:

$$
score(d) = \sum_{retrievers} \frac{1}{k+rank(d)}
$$
(with `k`= smoothening constant)
* Documents that appear across multiple queries/retrievers rise to the top.
* Reduces noise because only documents consistently relevant get boosted.

**Key Point:** RAG Fusion improves precision and robustness by rewarding cross-query/retriever consensus.

The diagram below illustrates this technique.

![Multi Query](../images/rag_fusion.png)

Below is the code to implement **RAG Fusion**. There will be a lot of duplicate code cells - this has been done intentionally to enable you to run 2 sections independently!

In [12]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

In [13]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [14]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
# No longer using Google Embeddings as I have apparently exhausted my free quota :(
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/text-embedding-004", task_type="retrieval_document"
# )
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_rag_weng"

In [15]:
def create_or_load_embeddings(
    embeddings, faiss_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        # text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        #     chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        # )
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [16]:
retriever = create_or_load_embeddings(embeddings, faiss_store)

Loading existing embeddings from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\advanced RAG\..\faiss_index_rag_weng


In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# from langchain_google_genai import ChatGoogleGenerativeAI

# RAG-Fusion: prompt
template = """You are a helpful assistant that generates multiple search queries 
based on a single input query.\n 
Return just a simple list of queries with no additional markup or text\n
Generate multiple search queries related to: {question} \n
Output ({num_queries} queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [18]:
generate_queries = (
    prompt_rag_fusion | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke(
    {
        "num_queries": 5,
        "question": "What is task decomposition for LLM agents?",
    }
)

['What is task decomposition for LLM agents?',
 'Task decomposition techniques for LLM agents',
 'How LLM agents use task decomposition',
 'Benefits of task decomposition in large language model agents',
 'Task decomposition strategies for autonomous LLM agents']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

In [19]:
from langchain.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    """Reciprocal_rank_fusion that takes multiple lists of ranked documents
    and an optional parameter k used in the RRF formula"""

    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key
            # (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

In [20]:
# Retrieve
from pydoc import doc

question = "What is task decomposition for LLM agents?"
# here we are firing multiple queries against the vector store, getting all the
# responses & creating a unique set from all the responses.
retrieval_chain_rf = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rf.invoke({"question": question, "num_queries": 5})
print(f"Got {len(docs)} documents")
# for i in range(5):
#     print(f"{docs[i]}\n\n")

# Note: docs -> List[(document, score)] (i.e. a list of tuples of (Document, score))
for i, (doc, score) in enumerate(docs[: len(docs) // 2]):
    print(f"Document #{i+1}\nContent: {doc.page_content}\nScore: {score}")

# # and print the retrival chain too
# console.print(f"Retrieval chain: {retrieval_chain}")

Got 8 documents
Document #1
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Score: 0.08279569892473118
Document #2
Content: The system comprises of 4 stages:
(1) Task planning: LLM works as the brain and parses the user requests into multiple tasks. There are four attributes associated with each task: task type, ID, dependencies, and arguments. They use few-shot examples to guide LLM to do task parsing and planning.
Score: 0.06557377049180328
Document #3
Content: Boiko et al. (2023) also looked into LLM-empowered agents for scientific discovery, to handle autonomous design, planning, and performance of complex scientific experiments. This agent can use tools to browse the Internet, read documentation, execute code, call robotics experimentation APIs and
Score: 0.0483870967741

In [21]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

final_rag_chain_rag_fusion = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = final_rag_chain_rag_fusion.invoke({"question": question, "num_queries": 5})
console.print(f"[blue]Question: {question}[/blue]\n")
console.print(f"[yellow]AI Response:[/yellow]")
console.print(Markdown(response))

Question: What is task decomposition for LLM agents?

AI Response:
For LLM agents, task decomposition is the process where the agent breaks down large, complex tasks into smaller, more manageable subgoals. This enables the efficient handling of complex tasks.

LLMs can perform task decomposition in several ways:

 1 With simple prompting: Using prompts like "Steps for XYZ." or "What are the subgoals for achieving XYZ?".            
 2 With task-specific instructions: For example, providing an instruction like "Write a story outline." for a novel-writ
 3 As part of task planning: The LLM acts as the "brain" to parse user requests into multiple tasks, each with attribute
 4 With human inputs.                                                                                                   


## Query Decomposition

**What is It?**
**Query decomposition** is a technique of breaking one complex user query into **several** _simpler_, _focused_ sub-queries; retrieving for each, and then combining the results. It’s usually done with an LLM:

1. **Input:** a long or multi-part user question.
2. **Decompose:** use an LLM prompt such as
    “Decompose this question into a list of simpler search queries.”
3. **Retrieve:** run each sub-query against your retriever/vector DB.
4. **Synthesize:** feed the retrieved chunks back into the LLM to build the final answer.

**Example**

User asks:

`Compare net profit trends and regulatory risks of Tesla over last 3 years.`

Decomposition may generate:
* "What are the net profit trends of Tesla in last 3 years?"
* "What are the regulatory risks faced by Tesla?"
* "How to compare these two aspects?"

**Key idea**
Break big task → smaller tasks → retrieve → combine.

**Subtle difference**
* **Multi-Query**: _same_ question phrased differently
* **Decomposition**: _different sub-questions_ covering _different aspects_.

**Why / When to Use Query Decomposition**

✅ Use it when:
* **Complex / multi-aspect questions:** 
    e.g., “Compare AutoGPT and BabyAGI, and explain how planning differs from memory.”

* **Broad tasks spanning sub-topics:**
    e.g., “Give me the pros/cons of hybrid search and explain when to use reciprocal rank fusion.”

* **Long, natural language queries:** with multiple clauses joined by “and”, “or”, “how … and also …”.

* **Poor retrieval recall:** when a single embedding search often misses pieces of the question.

🚫 Less helpful when:
* The query is **short and atomic** (e.g., “What is RAG Fusion?”).
* The corpus is tiny or each document already covers the entire topic.

Once the query is broken decomposed into individual queries, there are two broad techniques to retrieve responses:
* Answer individually
* Answer recursively

We'll cover decomposition & the two answering techniques in this section. You'll notice lot of repeating code to allow you to run this section independent of other sections.



In [23]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

In [24]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [25]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
# No longer using Google Embeddings as I have apparently exhausted my free quota :(
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/text-embedding-004", task_type="retrieval_document"
# )
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_rag_weng"

In [26]:
def create_or_load_embeddings(
    embeddings, faiss_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        # text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        #     chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        # )
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [27]:
retriever = create_or_load_embeddings(embeddings, faiss_store)

Loading existing embeddings from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\advanced RAG\..\faiss_index_rag_weng


As a first step, let us ask the LLM our question & check it's response without any query decomposition.

In [28]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [29]:
QUESTION = "What is task decomposition for LLM agents?"

In [30]:
template = ChatPromptTemplate.from_template(
    "Answer the following question:\n\n{question}"
)
simple_chain = template | llm | StrOutputParser()
response = simple_chain.invoke({"question": QUESTION})
console.print(Markdown(response))

Task decomposition for LLM agents is the process of breaking down a complex, overarching goal or task into a series of smaller, more manageable, and often sequential sub-tasks. Instead of trying to solve a large, multi-faceted problem in one go, the LLM agent first identifies and outlines the individual steps required to achieve the final objective.

Think of it like a project manager breaking down a large project into smaller sprints or individual assignments.

                                   Why is Task Decomposition Crucial for LLM Agents?                                    

LLMs, despite their impressive capabilities, have several limitations that task decomposition helps to mitigate:

 1 Complexity Handling: LLMs can struggle with long-chain reasoning and complex, multi-step problems. By breaking down t
 2 Reduced Hallucination: When faced with a very broad or complex prompt, LLMs are more prone to "hallucinating" or gene
 3 Context Window Management: LLMs have a limited conte

Now let's ask LLM to decompose our query as we discussed above.

In [31]:
# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related 
to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that 
can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Generate just the list of queries. Don't generate any other text, such as numbering or additional quotes around the queries or markdown text \n
Output ({num_queries} queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

In [32]:
queries_generator = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
questions = queries_generator.invoke(
    {
        "num_queries": 5,
        "question": QUESTION,
    }
)
console.print(questions)

[
    'task decomposition LLM agents definition',
    'how task decomposition improves LLM agent performance',
    'techniques for task decomposition in LLM agents',
    'benefits of task decomposition for large language model agents',
    'task decomposition frameworks for LLM-based agents'
]


### Answering Techniques for Query Decomposition
So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

Once we have the decomposed questions, we'll tweak the way the LLM responds to these questions.

#### Answering Individually
**How it works**:
* Break the big question into smaller sub-questions. 
* Retrieve documents **for each sub-question separately**. 
* Answer each sub-question independently using RAG.
* **Combine all sub-answers** into one final answer.

**Example**
_User asks_:

`Compare AI regulations in the US, EU, and China.`

_Possible Sub-questions generated_:

* "What are AI regulations in the US?"
* "What are AI regulations in the EU?"
* "What are AI regulations in China?"

_Process_:
* Retrieve for (1), answer (1)
* Retrieve for (2), answer (2)
* Retrieve for (3), answer (3)
* Then combine.

**Key Properties**
* Independent answers → high recall
* More context diversity
* Less chance of missing a sub-topic
* More expensive → many retrieval + LLM calls

**Subtle difference with _Anwer Recursively_** (which we will cover later)
* This method **does not use _intermediate answers_** to inform later ones.
* Each _sub-question is handled as if it is unrelated to the others_.

The image below illustrates this process visually:

![Multi Query](../images/ans_individually.png)

In [33]:
rag_prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.\n
    Question: {question} \n
    Context: {context} \n
    Answer:"""
)

In [34]:
def retrieve_individual_qna(
    question, prompt_rag, sub_question_generator_chain, num_queries=5
):
    # Use our decomposition /
    sub_questions = sub_question_generator_chain.invoke(
        {"question": question, "num_queries": num_queries}
    )

    # Initialize a list to hold RAG chain results
    rag_results = []

    for sub_question in sub_questions:
        # Retrieve documents for each sub-question
        retrieved_docs = retriever.get_relevant_documents(sub_question)
        # Use retrieved documents and sub-question in RAG chain
        answer = (prompt_rag | llm | StrOutputParser()).invoke(
            {"context": retrieved_docs, "question": sub_question}
        )
        rag_results.append(answer)

    return sub_questions, rag_results


def format_qna_pairs(questions, answers):
    """Format Q and A pairs"""

    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

In [35]:
questions, answers = retrieve_individual_qna(QUESTION, rag_prompt, queries_generator)

C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_2872\2699351271.py:14: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(sub_question)


In [37]:
qna_pairs = format_qna_pairs(questions, answers)
print(qna_pairs)

Question 1: task decomposition LLM agents definition
Answer 1: Task decomposition in LLM agents involves the agent breaking down large, complex tasks into smaller, manageable subgoals. The LLM acts as the core controller, parsing user requests into multiple tasks during the task planning stage. This process enables efficient handling of complex tasks and can be achieved by the LLM using simple prompting or few-shot examples.

Question 2: how task decomposition improves LLM agent performance
Answer 2: Task decomposition improves LLM agent performance by breaking down large tasks into smaller, manageable subgoals. This process enables the efficient handling of complex tasks. The LLM acts as the brain, parsing user requests into multiple tasks during task planning.

Question 3: techniques for task decomposition in LLM agents
Answer 3: Task decomposition in LLM agents can be achieved through several techniques. One method involves using the LLM itself with simple prompting, such as asking 

Now we feed each question & it's anwwer (i.e. complete output of previoius cell) as a context to LLM and ask it to extract answer from this context.

In [38]:
# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = prompt | llm | StrOutputParser()

response = final_rag_chain.invoke({"context": qna_pairs, "question": QUESTION})

console.print(f"[blue]Question: {question}[/blue]\n")
console.print(f"[yellow]AI Response:[/yellow]")
console.print(Markdown(response))

Question: What is task decomposition for LLM agents?

AI Response:
Task decomposition for LLM agents is the process where a large language model agent breaks down complex, overarching tasks into smaller, more manageable subgoals. The LLM acts as the core controller or "brain," responsible for parsing user requests and formulating these multiple subtasks during the task planning stage.

This approach is crucial for enabling the efficient handling of intricate problems and improving overall agent performance by making complex tasks more tractable.

Several techniques facilitate this decomposition:

 • LLM-driven Prompting: The LLM can decompose tasks autonomously using simple prompts, such as asking for "Steps for XY
 • Task-Specific Instructions: Pre-defined instructions tailored to the task, like "Write a story outline," can guide th
 • Human Input: Direct human guidance can also inform the breakdown process.                                            
 • Few-shot Examples: LLMs are of

#### Answering Recursively
In this technique, the questions list we got above is passed recursively to the LLM - first $Q_1$ is passed and we get a response $A_1$ from LLM. $Q_1$ + $A_1$ is added as a context to $Q_2$ to get $A_2$, then ($Q_1$ + $A_1$) and ($Q_2$ + $A_2$) is added as a context when passing $Q_3$ to the LLM and so on. Finally, we land up with context -> {($Q_1$ + $A_1$), ($Q_2$ + $A_2$), ..., ($Q_{N-1}$ + $A_{N-1}$) } when $Q_N$ is passed to the LLM. The answer from the LLM to QN with the above combined context is the final response. The intutition is by passing this "combined context" derived from the recursive process helps the LLM give a more coherent response to original question.

**Example**

User asks: 
* "Explain how a blockchain works and why it is secure".

Sub-questions:
* "How does a blockchain work?"
* "Why is it secure?"
* "Explain the connection between (1) and (2)."

**Process:**
* Retrieve + answer (1) 
* Pass answer (1) into answering (2) 
* Use (1) and (2) to explain (3)

**Key Properties**
* Answers accumulate → more reasoning continuity 
* Produces cohesive explanation
* Less duplication
* Less RAG calls (because previous answers seed later ones)

**Subtle difference**
* This method uses the **answer of each sub-step to inform the next**, creating a chain of reasoning — like a teacher building one idea after another.

The image below illustrates this process visually:

![Multi Query](../images/ans_recursively.png)

In [39]:
# Prompt
template = """Here is the question you need to answer:
\n --- \n {question} \n --- \n
Here is any available background question + answer pairs:
\n --- \n {q_a_pairs} \n --- \n
Here is additional context relevant to the question: 
\n --- \n {context} \n --- \n
Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [40]:
from operator import itemgetter


def format_qa_pair(question, answer):
    """Format Q and A pair"""
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()


q_a_pairs = ""
for i, q in enumerate(questions):
    rag_chain = (
        {
            # get the context by asking the retriver to retrieve it
            # based on the question
            "context": itemgetter("question") | retriever,
            "question": itemgetter("question"),
            # for the first question, q_a_pairs will be ""
            "q_a_pairs": itemgetter("q_a_pairs"),
        }
        # format my prompt with above parameters
        | decomposition_prompt
        # ask LLM for response to formatted decomposition prompt
        | llm
        # parse out text as answer
        | StrOutputParser()
    )
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair
    console.print(f"[yellow]Intermediate QA-Pair #{i+1} -> [/yellow]")
    console.print(Markdown(q_a_pairs))

Intermediate QA-Pair #1 -> 
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: task decomposition LLM agents definition Answer: In the context of LLM agents, task decomposition is the process where the agent breaks down large, complex tasks into smaller, more manageable subgoals. This capability is crucial for enabling the efficient handling of intricate problems.

The Large Language Model (LLM) acts as the core controller or "brain" for task planning, parsing user requests into these multiple tasks. Task decomposition can be achieved through various methods:

 1 LLM Prompting: Using simple prompts such as "Steps for XYZ" or "What are the subgoals for achieving XYZ?".           
 2 Task-Specific Instructions: Employing specific instructions tailored to the task, for example, "Write a story outline
 3 Human Inputs: Incorporating direct input from a human.                                                      

Notice how we keep adding a Q & A pair to the overall context. At the end of all the questions (Q&A pairs), we get the final answer from the LLM, which we will display below.

In [41]:
console.print(f"[yellow]Final answer:[/yellow]")
console.print(Markdown(answer))

Final answer:
Task decomposition frameworks for LLM-based agents primarily involve leveraging the LLM's capabilities as a core controller for task planning, often guided by various inputs. The main frameworks or methods include:

 1 LLM Prompting: The LLM itself can break down tasks by responding to simple prompts. Examples include:                
    • "Steps for XYZ."                                                                                                  
    • "What are the subgoals for achieving XYZ?" This allows the LLM to generate a sequence of steps or subgoals directl
 2 Task-Specific Instructions: Employing specific instructions tailored to the task at hand can guide the decomposition 
 3 Human Inputs: Direct input from a human can be incorporated to guide or perform the decomposition. This allows for ex
 4 Few-shot Examples: To guide the LLM in task parsing and planning, few-shot examples can be used. These examples help 

In these frameworks, the LLM acts as the "b

## Step Back
A different approach, presented by Google, is _Step-Back Prompting_. It takes the opposoite approach, where it tries to ask a more abstract question. So [the paper](https://arxiv.org/pdf/2310.06117.pdf) talks a lot about using few-shot prompting to produce what they call the _step-back_ (or more abstract) questions. The way it does it is to provide a number of examples of step-back questions, given the original question.

**Simple explanation**

Before answering directly, the LLM creates a more general version of the question → retrieves information at a high level → uses that to answer the original.

**Mental model**

Zoom out before zooming in.

**Example**

User asks: _"How do I implement a Kafka-based event-driven architecture for claims processing?"_

Step-back question:

* → “What is an event-driven architecture?”
* → Retrieve general knowledge 
* → Then answer the specific question.

Best for:
* Extremely narrow or specific queries
* When the vector store lacks specific content
* Knowledge-heavy topics where general understanding helps answer specifics

Differences:
* Unlike Multi-Query, Step-Back broadens the query instead of paraphrasing.
* Unlike Decomposition, it doesn’t split into parts but moves to higher abstraction.

![Step Back](../images/step_back.png)

In [42]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [43]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [44]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
# No longer using Google Embeddings as I have apparently exhausted my free quota :(
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/text-embedding-004", task_type="retrieval_document"
# )
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_rag_weng"

In [45]:
def create_or_load_embeddings(
    embeddings, faiss_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        # text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        #     chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        # )
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [46]:
retriever = create_or_load_embeddings(embeddings, faiss_store)

Loading existing embeddings from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\advanced RAG\..\faiss_index_rag_weng


Let's look a some _few-shot_ examples

In [47]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [48]:
generate_queries_step_back = prompt | llm | StrOutputParser()
question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

'What methods do LLM agents use?'

In [49]:
# Response prompt
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

response = chain.invoke({"question": question})
console.print(Markdown(response))

Task decomposition for LLM agents is a crucial process where a large, complex task is broken down into smaller, more manageable subgoals. This enables the agent to handle intricate problems efficiently and is a fundamental aspect of long-term planning.

Here's a breakdown of its key aspects for LLM agents:

 1 Purpose: The primary goal is to simplify complex user requests into actionable steps, making the overall task more tr
 2 LLM's Role as the "Brain": In agent systems, the Large Language Model (LLM) functions as the core controller or "brai
 3 Methods of Decomposition: Task decomposition can be achieved through several mechanisms:                             
    • LLM with Simple Prompting: The LLM can be prompted directly with instructions like "Steps for XYZ," or "What are t
    • Task-Specific Instructions: Pre-defined instructions tailored to specific types of tasks can guide the LLM in brea
    • Human Inputs: Human intervention can also be used to guide or directly provide t

### HYDE

**What is it?**

HYDE is an interesting approach that takes adbantage of a very simple idea. The basic RAG flow takes a question and embeds it; takes a document & embeds it and looks for similarity between an embeded document & the embedded question. However, the question & document are very dis-similar. A document can very large and complex - may come from _dense_ publications (such as PDFs) and other sources, whereas questions are usually short & terse and could be ill-worded from users.

The intuition behind HyDE is take questions and map them into document space using a hypothetical document (or by generating a hypothetical document). The idea is shown visually in the diagram below - in principle, for certain cases, a hypothetics document is _closer_ to desired document you want to retrieve from the high dimension vector space of the embedding than the sparse raw input question. This is a means of translating raw questions into hypotheticsl documents, which are better suited for retrieval.
 
![HYDE](../images/hyde.png)

In [50]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [51]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [52]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
# No longer using Google Embeddings as I have apparently exhausted my free quota :(
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/text-embedding-004", task_type="retrieval_document"
# )
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_rag_weng"

In [53]:
def create_or_load_embeddings(
    embeddings, faiss_store, chunk_size=300, chunk_overlap=50
):
    """creates (if not available) or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        # text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        #     chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        # )
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [54]:
retriever = create_or_load_embeddings(embeddings, faiss_store)

Loading existing embeddings from c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\advanced RAG\..\faiss_index_rag_weng


In [55]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = """Please write a scienticic paper passage to answer the following question:
Question: {question}
Passage: """

prompt_hyde = ChatPromptTemplate.from_template(prompt_template)

generate_docs_for_retrieval = prompt_hyde | llm | StrOutputParser()

question = "What is task decomposition for LLM Agents?"
generated_doc = generate_docs_for_retrieval.invoke({"question": question})
console.print(Markdown(generated_doc))


                                   Task Decomposition in Large Language Model Agents                                    

Task decomposition, in the context of Large Language Model (LLM) agents, refers to the process of breaking down a complex, high-level objective into a series of smaller, more manageable, and often sequential sub-tasks or sub-goals. This strategy mirrors human cognitive approaches to problem-solving, where intricate challenges are segmented into simpler, more tractable components, each of which can be addressed individually before being integrated into a complete solution.

The primary motivation for task decomposition stems from the inherent limitations of LLMs when confronted with multi-step reasoning, long-horizon planning, or tasks requiring extensive contextual understanding and memory over extended interactions. While LLMs excel at generating coherent text and performing single-step reasoning, their performance can degrade significantly on complex tasks due to

So we have generated a hypothetical document, which hopefully maps close to relevant documents in the larger vector embedding space. Now we can use this document to do a similarity search

In [56]:
retrieval_chain = generate_docs_for_retrieval | retriever
retrieved_docs = retrieval_chain.invoke({"question": question})
print(retrieved_docs)

[Document(id='ba9ef767-b88f-4518-8bdf-323595eddc4e', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.'), Document(id='ac302cb1-5bc5-48bb-a63a-d71ca6814fa6', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.'), Document(id='24053da6-1928-4785-b22b-b1ff28fa86b7', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, se

In [57]:
# build a context from generated docs for LLM to use to answer our original question
# build context
context = ""
for doc in retrieved_docs:
    context += doc.page_content + "\n\n"

And now you RAG from retrieved documents :)

In [ ]:
template = """Answer the following question based on the provided context.

{context}

Question: {question}"""

prompt_template = ChatPromptTemplate.from_template(template)

final_chain = prompt_template | llm | StrOutputParser()
final_response = final_chain.invoke({"context": context, "question": question})
console.print(f"[blue]Question: {question}[/blue]\n")
console.print(f"[yellow]AI Response:[/yellow]")
console.print(Markdown(final_response))

Question: What is task decomposition for LLM Agents?

AI Response:
Task decomposition for LLM Agents is the process where the agent breaks down large, complex tasks into smaller, more manageable subgoals. This enables efficient handling of these complex tasks.

It can be achieved in a few ways:

 1 By the LLM itself with simple prompting (e.g., "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?").    
 2 By using task-specific instructions (e.g., "Write a story outline." for writing a novel).                            
 3 With human inputs.                                                                                                   

In the system described, the LLM works as the "brain" during the "Task planning" stage, parsing user requests into multiple tasks, which is a form of decomposition.


### Summary

Above, we saw various techniques to transform user-queries at the head of the RAG pipeline. The table below summarizes all these approaches for quick reference:

| Technique | Use When | Avoid When |
|:----------|:---------|:-----------|
| Multi-Query | User query is vague/ambiguous. Terminology varies across documents. | If query is already precise. |
| RAG Fusion | You want highest accuracy. There are multiple retrieval signals. | Small datasets (not enough data to fuse).|
| Query Decomposition | Query is long, multi-part, or analytical. | Simple single-topic questions. |
| Step Back | Query is overly narrow, domain-specific, or uncommon. | When generalization hurts precision. |
| HYDE | Documents are sparse. Query is short/speculative. |  If vector store is dense and high-quality. | 

How to determine in which "class" a user's query falls at run-time is indeed an interesting challenge, which we will cover in a separate workbook.